# Early Disease Detection: Cardiovascular Disease Prediction

This notebook builds a classification model to predict the presence of heart disease based on health and demographic indicators.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

## 1. Data Exploration and Preprocessing (EDA)

In [ ]:
df = pd.read_csv('heart_disease_data.csv')
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

### Missing Values and Data Integrity
Let us check for nulls and anomalies.

In [ ]:
print('Missing values:\n', df.isnull().sum())

# Check for negative or unrealistic blood pressures
weird_bp = df[(df.ap_hi < 50) | (df.ap_hi > 250) | (df.ap_lo < 30) | (df.ap_lo > 200)]
print(f'\nFound {len(weird_bp)} records with unusual blood pressure. These will be capped or filtered out.')

In [ ]:
plt.figure(figsize=(6,4))
sns.countplot(x='disease', data=df)
plt.title('Distribution of Heart Disease (Target Variable)')
plt.show()

### Preprocessing Feature Transformations
- Calculate ge in years (currently in days).
- Filter out extreme anomalies in blood pressure.
- Drop unnecessary identifiers (date, id).
- One-hot encode categorical demographic features (country, occupation).

In [ ]:
# 1. Convert age (drops fraction for integer years)
df['age_years'] = (df['age'] / 365.25).astype(int)

# 2. Filter anomalous blood pressure values (keep realistic ranges)
df = df[(df.ap_hi >= 50) & (df.ap_hi <= 250)]
df = df[(df.ap_lo >= 30) & (df.ap_lo <= 200)]

# 3. Drop unneeded columns
df_clean = df.drop(['id', 'date', 'age'], axis=1)

# 4. One-hot encoding for categorical variables
df_clean = pd.get_dummies(df_clean, columns=['country', 'occupation'], drop_first=True)

df_clean.head()

## 2. Model Development
We split the dataset 80/20 and apply Standard Scaling as models like SVM and Logistic Regression are distance/scale sensitive.

In [ ]:
X = df_clean.drop('disease', axis=1)
y = df_clean['disease']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale numeric features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Training set size: {X_train_scaled.shape}')
print(f'Testing set size: {X_test_scaled.shape}')

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=10),
    'Linear SVC': LinearSVC(random_state=42, max_iter=2000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10)
}

results = {}

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results[name] = [acc, prec, rec, f1]
    
print('Training Complete!')

## 3. Model Evaluation
Comparing the classification metrics of the models.

In [ ]:
results_df = pd.DataFrame(results, index=['Accuracy', 'Precision', 'Recall', 'F1-Score']).T
print(results_df)

results_df.plot(kind='bar', figsize=(10, 6))
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.ylim(0, 1.0)
plt.legend(loc='lower right')
plt.show()

In [ ]:
best_model = models['Random Forest']
y_pred_best = best_model.predict(X_test_scaled)

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix (Random Forest)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 4. Insights and Reporting
What features drive the prediction? We can look at the Random Forest feature importances.

In [ ]:
importances = best_model.feature_importances_
feature_names = X.columns

feat_imp_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
feat_imp_df = feat_imp_df.sort_values(by='Importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feat_imp_df)
plt.title('Top 15 Most Important Features for Predicting Heart Disease')
plt.show()